In [ ]:
!wget https://zhanggroup.org/BioLiP/download/BioLiP_nr.txt.gz -O downloads/biolip.txt.gz
!wget https://zhanggroup.org/BioLiP/data/protein_nr.fasta.gz -O downloads/biolip.fasta.gz

.downloads/biolip.txt.gz: No such file or directory
--2025-09-10 15:06:36--  https://zhanggroup.org/BioLiP/data/protein_nr.fasta.gz
Resolving zhanggroup.org (zhanggroup.org)... 137.132.93.250
Connecting to zhanggroup.org (zhanggroup.org)|137.132.93.250|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8021454 (7.6M) [application/x-gzip]
Saving to: ‘.downloadsbiolip.fasta.gz’

.downloadsbiolip.fa 100%[===================>]   7.65M   319KB/s    in 24s     

2025-09-10 15:07:01 (321 KB/s) - ‘.downloadsbiolip.fasta.gz’ saved [8021454/8021454]



/tmp/ipykernel_2726076/3314446359.py:2: DtypeWarning: Columns (13,14,16) have mixed types. Specify dtype option on import or set low_memory=False.
  biolip_df = pd.read_csv('downloads/biolip.txt', sep='\t', header=None)


In [ ]:
import pandas as pd
biolip_df = pd.read_csv('downloads/biolip.txt', sep='\t', header=None)
# reasonable column names for list above
biolip_df.columns = [
    'pdb_id', 'receptor_chain', 'resolution', 'binding_site_number_code',
    'ligand_id', 'ligand_chain', 'ligand_serial_number', 'binding_site_residues_pdb_num',
    'binding_site_residues_renumbered', 'catalytic_site_residues_pdb_num',
    'catalytic_site_residues_renumbered', 'ec_number', 'go_terms',
    'binding_affinity_manual_survey', 'binding_affinity_binding_moad',
    'binding_affinity_pdbbind_cn', 'binding_affinity_bindingdb',
    'uniprot_id', 'pubmed_id', 'ligand_residue_sequence_number',
    'receptor_sequence'
]
biolip_df.dropna(subset=['pdb_id', 'receptor_chain'], inplace=True)
# biolip_df['seq_len'] = biolip_df['receptor_sequence'].apply(len)
biolip_df['fasta_id'] = biolip_df['pdb_id'].str.cat(biolip_df['receptor_chain'].str.strip())

#load biolip fasta file with biopython
from Bio import SeqIO
fasta_sequences = SeqIO.parse(open('downloads/biolip.fasta'), 'fasta')
fasta_dict = {record.id: str(record.seq) for record in fasta_sequences}
biolip_df = biolip_df[biolip_df['fasta_id'].isin(fasta_dict)]
biolip_df['protein_sequence'] = biolip_df['fasta_id'].map(fasta_dict)
biolip_df['seq_len'] = biolip_df['protein_sequence'].apply(len)

/tmp/ipykernel_2726076/3780536591.py:2: DtypeWarning: Columns (13,14,16) have mixed types. Specify dtype option on import or set low_memory=False.
  biolip_df = pd.read_csv('downloads/biolip.txt', sep='\t', header=None)


In [80]:
biolip_df['ligand_id'].value_counts().head(20)
#select 200 random rows from each of the top 10 ligands and save to new dataframe
top_20_ligands = biolip_df['ligand_id'].value_counts().head(12).index.tolist()
biolip_top_20 = biolip_df[biolip_df['ligand_id'].isin(top_20_ligands)].groupby('ligand_id').apply(lambda x: x.sample(n=200, random_state=1) if len(x) >= 200 else x).reset_index(drop=True)
biolip_top_20 = biolip_top_20[biolip_top_20['seq_len'] < 850]

/tmp/ipykernel_2726076/2789014906.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  biolip_top_20 = biolip_df[biolip_df['ligand_id'].isin(top_20_ligands)].groupby('ligand_id').apply(lambda x: x.sample(n=200, random_state=1) if len(x) >= 200 else x).reset_index(drop=True)


In [78]:
def get_binding_side_residues(res_renum_str):
    res = res_renum_str.split(' ')
    res_ind = [int(r[1:]) for r in res]
    return res_ind
def get_binding_site_AA(res_renum_str):
    res = res_renum_str.split(' ')
    res_aa = [r[0] for r in res]
    return res_aa
def get_site_AA(res_ind, seq):
    site_aa = [seq[i-1] for i in res_ind if i-1 < len(seq)]
    return site_aa
biolip_top_20['binding_site_residues_list'] = biolip_top_20['binding_site_residues_renumbered'].apply(get_binding_side_residues)
biolip_top_20['binding_site_AA'] = biolip_top_20['binding_site_residues_renumbered'].apply(get_binding_site_AA)
biolip_top_20['binding_site_AA_seq'] = biolip_top_20.apply(lambda row: get_site_AA(row['binding_site_residues_list'], row['protein_sequence']), axis=1)

In [82]:
biolip_top_20['go_terms']
biolip_top_20['go_terms'] = biolip_top_20['go_terms'].fillna('').apply(lambda x: x.split(',') if x != '' else [])
biolip_top_20['go_terms'] = biolip_top_20['go_terms'].apply(lambda x: ['GO:' + term.zfill(7) if not term.startswith('GO:') else term for term in x])

In [83]:
biolip_top_20['go_terms']

0       [GO:0003677, GO:0005515, GO:0005524, GO:000626...
2       [GO:0003991, GO:0005524, GO:0005737, GO:000652...
3       [GO:0000492, GO:0000723, GO:0000786, GO:000081...
4        [GO:0003777, GO:0005524, GO:0007018, GO:0008017]
5       [GO:0000166, GO:0003824, GO:0004356, GO:000654...
                              ...                        
2394     [GO:0003735, GO:0005840, GO:0006412, GO:1990904]
2395    [GO:0003735, GO:0005840, GO:0006412, GO:000950...
2397    [GO:0000398, GO:0000974, GO:0003723, GO:000551...
2398    [GO:0000166, GO:0003677, GO:0003735, GO:000584...
2399                                                   []
Name: go_terms, Length: 2234, dtype: object

In [ ]:
columns = ['UniprotID', 'AnnotatedIndices', 'GOTerm', 'Sequence']
col_map = {
    'UniprotID': 'uniprot_id',
    'AnnotatedIndices': 'binding_site_residues_renumbered',
    'GOTerm': 'go_terms',
    'Sequence': 'protein_sequence'}

In [79]:
biolip_top_20

,pdb_id,receptor_chain,resolution,binding_site_number_code,ligand_id,ligand_chain,ligand_serial_number,binding_site_residues_pdb_num,binding_site_residues_renumbered,catalytic_site_residues_pdb_num,...,uniprot_id,pubmed_id,ligand_residue_sequence_number,receptor_sequence,fasta_id,protein_sequence,seq_len,binding_site_residues_list,binding_site_AA,binding_site_AA_seq
0,6qem,G,3.400,BS02,ADP,G,1,H66 Y74 R75 P108 G109 T110 G111 K112 N113 Y236...,H66 Y74 R75 P108 G109 T110 G111 K112 N113 Y236...,NaN,...,P0AEF0,30797687.0,301,MKNVGDLMQRLQKMMPAHIKPAFKTGEELLAWQKEQGAIRSAALER...,6qemG,MKNVGDLMQRLQKMMPAHIKPAFKTGEELLAWQKEQGAIRSAALER...,239,"[66, 74, 75, 108, 109, 110, 111, 112, 113, 236...","[H, Y, R, P, G, T, G, K, N, Y, R]","[H, Y, R, P, G, T, G, K, N, Y, R]"
2,2rd5,B,2.510,BS02,ADP,B,1,A45 T215 D216 L221 K224 A249 G251 M252 K255,A31 T201 D202 L207 K210 A235 G237 M238 K241,K41 G44 G77 D196 K255,...,Q9SCL7,17913711.0,2000,SPDYRVEILSESLPFIQKFRGKTIVVKYGGAAMTSPELKSSVVSDL...,2rd5B,SPDYRVEILSESLPFIQKFRGKTIVVKYGGAAMTSPELKSSVVSDL...,283,"[31, 201, 202, 207, 210, 235, 237, 238, 241]","[A, T, D, L, K, A, G, M, K]","[A, T, D, L, K, A, G, M, K]"
3,7zi4,C,3.200,BS01,ADP,C,1,S17 H18 H20 G38 V40 G73 G75 K76 T77 A78 Y366 R404,S16 H17 H19 G37 V39 G72 G74 K75 T76 A77 Y359 R397,NaN,...,Q9Y265,NaN,501,KIEEVKSTTKTQRIASHSHVKGLGLDESGLAKQAASGLVGQENARE...,7zi4C,KIEEVKSTTKTQRIASHSHVKGLGLDESGLAKQAASGLVGQENARE...,447,"[16, 17, 19, 37, 39, 72, 74, 75, 76, 77, 359, ...","[S, H, H, G, V, G, G, K, T, A, Y, R]","[S, H, H, G, V, G, G, K, T, A, Y, R]"
4,3x2t,A,2.700,BS01,ADP,A,1,R14 R16 P17 T88 S89 G91 K92 T93 H94,R11 R13 P14 T85 S86 G88 K89 T90 H91,NaN,...,P28738,25777528.0,2000,PAECSIKVMCRFRPLNEAEILRGDKFIPKFKGEETVVIGQGKPYVF...,3x2tA,PAECSIKVMCRFRPLNEAEILRGDKFIPKFKGEETVVIGQGKPYVF...,311,"[11, 13, 14, 85, 86, 88, 89, 90, 91]","[R, R, P, T, S, G, K, T, H]","[R, R, P, T, S, G, K, T, H]"
5,7cqw,A,2.297,BS01,ADP,A,1,Y174 Y191 H237 L238 S239 R317 R323,Y174 Y191 H237 L238 S239 R317 R323,NaN,...,A0A369R1N0,33199371.0,500,MTDLASIAREKGIEFFLISFTDLLGVQRAKLVPARAIADMAVNGAG...,7cqwA,MTDLASIAREKGIEFFLISFTDLLGVQRAKLVPARAIADMAVNGAG...,430,"[174, 191, 237, 238, 239, 317, 323]","[Y, Y, H, L, S, R, R]","[Y, Y, H, L, S, R, R]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2394,7olc,SM,2.900,BS01,rna,2,0,R45 G46 R48 E49 K52 C68 E69 E70 Y73 C118,R21 G22 R24 E25 K28 C44 E45 E46 Y49 C94,NaN,...,G0SHH9,35079002.0,1 ~ 1796,SVLDALKGVLKLSLMHDGLARGLREASKALDRRQAHMCVLNESCEE...,7olcSM,SVLDALKGVLKLSLMHDGLARGLREASKALDRRQAHMCVLNESCEE...,118,"[21, 22, 24, 25, 28, 44, 45, 46, 49, 94]","[R, G, R, E, K, C, E, E, Y, C]","[R, G, R, E, K, C, E, E, Y, C]"
2395,5mmi,Y,3.200,BS01,rna,A,0,R72 R73 K80 N83 K84 A85 N86 R87 S89 S91 N92 H9...,R1 R2 K9 N12 K13 A14 N15 R16 S18 S20 N21 H22 K...,NaN,...,P82245,28007896.0,2 ~ 2810,RRICPFTGKKSNKANRVSHSNHKTKRLQFVNLQYKRVWWEAGKRFV...,5mmiY,RRICPFTGKKSNKANRVSHSNHKTKRLQFVNLQYKRVWWEAGKRFV...,77,"[1, 2, 9, 12, 13, 14, 15, 16, 18, 20, 21, 22, ...","[R, R, K, N, K, A, N, R, S, S, N, H, K, R, Q, ...","[R, R, K, N, K, A, N, R, S, S, N, H, K, R, Q, ..."
2397,9fmd,P,3.300,BS02,rna,5,0,S32 R33 P36 S37 H38 T39,S31 R32 P35 S36 H37 T38,NaN,...,Q9P013,38925148.0,9 ~ 115,TTAARPTFEPARGGRGKGEGDLSQLSKQYSSRDLPSHTKIKYRQTT...,9fmdP,TTAARPTFEPARGGRGKGEGDLSQLSKQYSSRDLPSHTKIKYRQTT...,106,"[31, 32, 35, 36, 37, 38]","[S, R, P, S, H, T]","[S, R, P, S, H, T]"
2398,4v9f,Y,2.404,BS01,rna,0,0,R115 Q119 R120 H121 R122 K125 P126 Q127 F128 R...,R21 Q25 R26 H27 R28 K31 P32 Q33 F34 R36 Q37 D3...,NaN,...,P12736,23695244.0,8 ~ 2917,TELQARGLTEKTPDLSDEDARLLTQRHRVGKPQFNRQDHHKKKRVS...,4v9fY,TELQARGLTEKTPDLSDEDARLLTQRHRVGKPQFNRQDHHKKKRVS...,144,"[21, 25, 26, 27, 28, 31, 32, 33, 34, 36, 37, 3...","[R, Q, R, H, R, K, P, Q, F, R, Q, D, H, K, K, ...","[R, Q, R, H, R, K, P, Q, F, R, Q, D, H, K, K, ..."


/tmp/ipykernel_2726076/3269272044.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  biolip_top_20 = biolip_df[biolip_df['ligand_id'].isin(top_20_ligands)].groupby('ligand_id').apply(lambda x: x.sample(n=200, random_state=1) if len(x) >= 200 else x).reset_index(drop=True)


In [ ]:
# print(list(fasta_dict.keys())[:20])

In [60]:
print(biolip_df)

      pdb_id receptor_chain  resolution binding_site_number_code ligand_id  \
0       10mh              A        2.55                     BS01       dna   
1       10mh              A        2.55                     BS02       dna   
2       10mh              A        2.55                     BS03       SAH   
3       11as              A        2.50                     BS01       ASN   
4       11ba              A        2.06                     BS01       UPA   
...      ...            ...         ...                      ...       ...   
86453   9vid              B        2.22                     BS01       HEM   
86454   9vmx              D        3.20                     BS01   peptide   
86455   9vmx              D        3.20                     BS02       L9Q   
86456   9vmx              D        3.20                     BS03       L9Q   
86457   9vqm              A        3.50                     BS01        CU   

      ligand_chain  ligand_serial_number  \
0                B 

In [38]:
biolip_top_20.head()

,pdb_id,receptor_chain,resolution,binding_site_number_code,ligand_id,ligand_chain,ligand_serial_number,binding_site_residues_pdb_num,binding_site_residues_renumbered,catalytic_site_residues_pdb_num,...,go_terms,binding_affinity_manual_survey,binding_affinity_binding_moad,binding_affinity_pdbbind_cn,binding_affinity_bindingdb,uniprot_id,pubmed_id,ligand_residue_sequence_number,receptor_sequence,seq_len
0,2i5b,A,2.800,BS01,ADP,A,1,N141 T178 G180 G181 K182 A188 D190 I206 A214 G...,N139 T176 G178 G179 K180 A186 D188 I204 A212 G...,T178 T211 G213 A214 G215 C216,...,"0005524,0005829,0008478,0008902,0008972,000922...",NaN,NaN,NaN,NaN,P39610,16978644.0,301,MHKALTIAGSDSSGGAGIQADLKTFQEKNVYGMTALTVIVAMDPNN...,269
1,2dhr,A,3.900,BS01,ADP,A,1,G199 V200 G201 K202 T203 H204 I334 H338,G57 V58 G59 K60 T61 H62 I192 H196,NaN,...,"0004176,0004222,0005524,0006508,0016020,0016887",NaN,NaN,NaN,NaN,Q5SI82,16762831.0,1001,RARVLTEAPKVTFKDVAGAEEAKEELKEIVEFLKNPSRFHEMGARI...,458
2,8q72,A,4.170,BS01,ADP,A,1,G46 S47 G48 K49 T50 T51 D77 R78 S88 P90 G91 R1070,G46 S47 G48 K49 T50 T51 D77 R78 S88 P90 G91 R725,NaN,...,NaN,NaN,NaN,NaN,NaN,A0A6D0I2P0,38309275.0,1101,MNQVSGLAGKESFILTRIELFNWGGFHGLHQAAIHQDGTAVIGPTG...,750
3,4yj1,A,2.050,BS01,ADP,A,1,P263 L273 A277 V278 G315 F316 H317 G318 K319 S...,P208 L218 A222 V223 G260 F261 H262 G263 K264 S...,NaN,...,"0000166,0000963,0003723,0003729,0005737,000573...",NaN,Kd=885nM,NaN,NaN,Q57ZF2,26117548.0,701,GFSPLMDFFHSVEGRNYGELRSLTNETYQISENVRCTFLSIQSDPF...,600
4,3ruq,A,2.798,BS01,ADP,A,1,L39 G40 P41 G92 T94 T95 K161 G403 G404 V488 E490,L36 G37 P38 G89 T91 T92 K158 G400 G401 V485 E487,D60 T93 T94 D386,...,"0005524,0006457,0016887,0042802,0046872,005108...",NaN,NaN,NaN,NaN,Q877G8,22193720.0,545,QPGVLPENMKRYMGRDAQRMNILAGRIIAETVRSTLGPKGMDKMLV...,516
